In [ ]:
# ====================================
# Import libraries
# ====================================
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

#Nb. I have tried to re-use the code for the classification task as much
#as I could for consistency. I have flagged the spots where something changes with "NEW".

In [ ]:
# ====================================
# Reproducibility and device
# ====================================
r_seed = 99
random.seed(r_seed)
np.random.seed(r_seed)
torch.manual_seed(r_seed)
generator = torch.Generator().manual_seed(r_seed)

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(f"Using device: {device}")

In [ ]:
# ====================================
# Load the Clock dataset (75x75)
# ====================================
images_path = "data/075_images.npy"
labels_path = "data/075_labels.npy"

images_np = np.load(images_path)
labels_np = np.load(labels_path)

#Check out your files 
print("images:", images_np.shape, images_np.dtype, images_np.min(), images_np.max()) #18'000 images, 75x75
print("labels:", labels_np.shape, labels_np.dtype) #18'000 labels, 2 dimensions (hour, min)
print("hours:", np.unique(labels_np[:, 0])) #they go from 0 to 11
print("minutes:", labels_np[:, 1].min(), labels_np[:, 1].max()) #they go from 0 to 59

#Convert into pytorch tensors
images = torch.from_numpy(images_np).float() / 255.0 #turn numpy array into pytorch tensor, float, and rescale between 0 and 1
images = images.unsqueeze(1) #add channel dimension, we need format (batch_size, channels, height, width) (greyscale = 1 channel)

labels_minutes = torch.from_numpy(labels_np[:, 0] * 60 + labels_np[:, 1]).float() #NEW: hours*60 + minutes, and turn it into a float number
print(labels_minutes.shape, labels_minutes.min(), labels_minutes.max()) #18'000 labels, 1 dimension (min). Minutes go from 0 (00:00) to 719 (11:59). 

#What's new:
#label_minutes is now a float number (e.g. 90.5 minutes)

In [ ]:
#=====================================
# Train-Validation-Test split
# ====================================
dataset = TensorDataset(images, labels_minutes)
n = len(dataset)

train_size = int(0.8 * n)
valid_size = int(0.1 * n)
test_size = n - train_size - valid_size

train_data, valid_data, test_data = random_split(
    dataset,
    [train_size, valid_size, test_size],
    generator=generator
)

batch_size = 64

train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_data,
    batch_size=batch_size,
    shuffle=False 
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# =====================================
# Setting up the CNN for regression
# =====================================

class Net(nn.Module):

    def __init__(self): #allow to provide a desired number of classes
        super().__init__()
        #Input image (1, 75, 75) (greyscale, 75x75)
        self.conv1 = nn.Conv2d(1, 6, 3) #convolutional layer 1: 6 filters, 3x3
        self.pool = nn.MaxPool2d(2, 2) #pooling to reduce dimension: 6 filters, 36x36
        self.conv2 = nn.Conv2d(6, 16, 3) #convolutional layer 2: 16 filters, 3x3 (input 6 filters 36x36)
        self.fc1 = nn.Linear(16 * 17 * 17, 120) #fully connected layer: receives 16 filters, pooled to 17x17
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 1) #NEW: now we want only 1 output value

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) #Step 1: conv1 --> relu --> maxpool
        x = self.pool(F.relu(self.conv2(x))) #Step 2: conv2 --> relu --> maxpool
        x = torch.flatten(x, 1) #Step 3: flatten
        x = F.relu(self.fc1(x)) #Step 4: fully connected 1 --> relu
        x = F.relu(self.fc2(x)) #Step 5: fully connected 2 --> relu
        x = self.fc3(x) #Step 6: output

        return x
    
#What's new?
#Now our NN needs to output one number, not n (num_classes) classes.

In [ ]:
# ======================================================
# Defining the functions for training and evaluation
# ======================================================


#Train the model

def train_model(model, train_dataloader, optimizer, criterion, epochs): #NEW: no more bin_size
    
    """Train a model for a given number of epochs."""
    
    for epoch in range(epochs):
        
        model.train() 
        running_loss = 0.0
        
        for inputs, minutes in train_dataloader: 

            #zero the parameter gradients
            optimizer.zero_grad()

            #forward + backward + optimize
            outputs = model(inputs).squeeze(1) #NEW: these are still the predictions, but 
            loss = criterion(outputs, minutes) #NEW: regression loss  
            loss.backward() #compute gradients
            optimizer.step() #optimize weights

            #compute running loss
            running_loss += loss.item()
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"- Loss = {running_loss/len(train_dataloader):.4f}"
        )

#Validate the model

def evaluate_model(model, dataloader): #NEW: no more bin_size

    """Evaluate a model using the average circular time error."""

    model.eval()

    total_error = 0
    total = 0

    with torch.no_grad():
        for inputs, minutes in dataloader:
            
            #Get predictions (in total number of minutes)
            outputs = model(inputs).squeeze(1) #NEW: predictions (in minutes!!!)
            predicted_minutes = torch.remainder(outputs, 720) #NEW: restrict prediction in (0, 720) minutes range

            #Calculate absolute difference and the circular difference
            difference = torch.abs(predicted_minutes - minutes)
            circular_difference = torch.minimum(difference, 720 - difference)

            #Compute the errors
            total_error += circular_difference.sum().item()
            total += minutes.size(0)

    common_sense_error = total_error / total

    return common_sense_error

In [ ]:

# =====================================
# Regression experiments - to choose lr and epochs
# =====================================

experiments = [
    {"learning_rate": 0.001, "epochs": 10},
    {"learning_rate": 0.001, "epochs": 20},
    {"learning_rate": 0.0005, "epochs": 10},
    {"learning_rate": 0.0005, "epochs": 20}
]

results = []

for experiment in experiments:

    lr = experiment["learning_rate"]
    epochs = experiment["epochs"]

    print(f"\nExperiment: learning rate = {lr}, epochs = {epochs}")

    #New model for every experiment
    model = Net()

    #Loss function
    criterion = nn.MSELoss()

    #Optimizer
    optimizer = optim.Adam(model.parameters(), lr=lr)

    #Train the model
    train_model(
        model,
        train_dataloader,
        optimizer,
        criterion,
        epochs
    )

    #Validate the model
    valid_error = evaluate_model(model, valid_dataloader)

    #Store results
    results.append({
        "Learning rate": lr,
        "Epochs": epochs,
        "Validation error (min)": valid_error
    })

    print(f"Validation error: {valid_error:.2f} minutes")

results_df = pd.DataFrame(results)
print(results_df)


In [ ]:

# =====================================
# Regression experiments - to choose Loss Function
# =====================================

#Best parameters (according to validation results)
best_lr = 0.001
best_epochs = 20

#Loss function
loss_functions = {
    "MSELoss": nn.MSELoss(),
    "L1Loss": nn.L1Loss(),
    "SmoothL1Loss (beta = 5)": nn.SmoothL1Loss(beta=5.0)
}

for loss_name, criterion in loss_functions.items():

    print(f"\nTraining with {loss_name}")

    torch.manual_seed(99)

    #New model for each experiment
    final_model = Net()

    #Optimizer
    optimizer = optim.Adam(final_model.parameters(), lr=best_lr)

    #Train the model
    train_model(
        model,
        train_dataloader,
        optimizer,
        criterion,
        epochs
    )

    #Validate the model
    validation_error = evaluate_model(model, valid_dataloader)

    #Store results
    results.append({
        "Loss function": loss_name,
        "Validation error (minutes)": validation_error
    })

    print(f"{loss_name} validation error: {validation_error:.2f} minutes")


In [ ]:

# =====================================
# Test set - With Best Parameters
# =====================================

#Best parameters (according to validation results)
best_lr = 0.001
best_epochs = 20

#Final model
final_model = Net()

#Loss function
criterion = nn.MSELoss() #best loss function

#Optimizer
optimizer = optim.Adam(final_model.parameters(), lr=best_lr)

#Train on training data
train_model(
    final_model,
    train_dataloader,
    optimizer,
    criterion,
    best_epochs
)

#Evaluate the model on the test set
test_error = evaluate_model(
    final_model, 
    test_dataloader
)

print(f"Final test common-sense error: {test_error:.2f} minutes")


In [ ]:
# =====================================
# Questions
# =====================================

"""
What loss functions (nn.MSELoss, nn.L1Loss, nn.SmoothL1Loss) are appropriate?

- nn.MSELoss: computes the mean squared error between predictions and target y.

- nn.L1Loss (MAELoss): computes the mean absolute error between predictions and target y.

- nn.SmoothL1Loss: combines MSELoss and L1Loss by using a transition threshold (beta). For errors smaller 
then delta we use the MSELoss, for errors larger than beta we use the MAELoss.

The MSE loss penalises strongly the magnitude of the errors by taking their squared value, therefore large errors 
get a very high penalty. MAE, on the other hand, is less sensitive to outliers, and penalises the errors proportionally.
Lastly Smooth Loss is an option in the middle: it applies the MSE penalty to errors up until 5 minutes (if beta = 5.0), 
so the penalties grow a lot when the error is between 1 minute and 5 minutes, but then grow linearly when the error 
is more than 5 minutes. It's a bit less aggressive then MSE. 

What boundary discontinuity issues arise around 12:00 / 0:00?

The problem that arises at the boundary 12:00 to 00:00 is the fact that two predictions such as 11:59 
and 00:02 are only 3 minutes apart, but when we convert them into total minutes 11:59 = 11*60 + 59 = 718 minutes
and 00:02 = 0*60 + 2 = 2 minutes they seem to be 718 - 2 = 716 minutes apart. 

This is due to the circular nature of time. We handled this problem in the validation function by taking
circular_difference = torch.minimum(difference, 720 - difference) before we compute the "common sense error".
However, the loss function does not know this (just like the Cross entropy function didn't know).

"""